# Unit 5 – Jaccard Coefficient Calculations
**Module:** Machine Learning (ML_PCOM7E)  |  **e-Portfolio activity:** Jaccard Coefficient Calculations

**Task:** the pathological test table below gives results for three individuals. Calculate the Jaccard coefficient for (Jack, Mary), (Jack, Jim) and (Jim, Mary).

| Name | Gender | Fever | Cough | Test-1 | Test-2 | Test-3 | Test-4 |
|---|---|---|---|---|---|---|---|
| Jack | M | Y | N | P | N | N | A |
| Mary | F | Y | N | P | A | P | N |
| Jim  | M | Y | P | N | N | N | A |

**Method.** Jaccard similarity for two objects described by *asymmetric* binary attributes is

J(i, j) = q / (q + r + s)

where q = attributes that are 1 for both, r = 1 for i and 0 for j, s = 0 for i and 1 for j. Attributes that are 0 for both (the count *t*) are ignored.

**Assumptions (stated because the table does not define the codes).**
1. *Gender* is a symmetric attribute (M and F are equally informative), so it is excluded from the asymmetric Jaccard calculation.
2. For the remaining attributes the "positive / present" value is coded 1: **Y** and **P** = 1; **N** and **A** = 0 (A read as *absent*).
3. Section 4 repeats the calculation with **A = 1** to show how sensitive the ranking is to assumption 2.

In [1]:
import pandas as pd, itertools
df = pd.DataFrame({"Name":["Jack","Mary","Jim"],"Gender":["M","F","M"],"Fever":list("YYY"),"Cough":list("NNP"),
                   "Test-1":list("PPN"),"Test-2":list("NAN"),"Test-3":list("NPN"),"Test-4":list("ANA")}).set_index("Name")
df

,Gender,Fever,Cough,Test-1,Test-2,Test-3,Test-4
Name,,,,,,,
Jack,M,Y,N,P,N,N,A
Mary,F,Y,N,P,A,P,N
Jim,M,Y,P,N,N,N,A


## 1. Encode the asymmetric attributes

In [2]:
def encode(frame, positive=("Y","P")):
    return frame.drop(columns="Gender").apply(lambda col: col.isin(positive).astype(int))
B = encode(df); B

,Fever,Cough,Test-1,Test-2,Test-3,Test-4
Name,,,,,,
Jack,1,0,1,0,0,0
Mary,1,0,1,0,1,0
Jim,1,1,0,0,0,0


## 2. Jaccard coefficient with the working shown

In [3]:
def jaccard_detail(a, b):
    q = int(((a==1)&(b==1)).sum()); r = int(((a==1)&(b==0)).sum()); s = int(((a==0)&(b==1)).sum()); t = int(((a==0)&(b==0)).sum())
    return q, r, s, t, q/(q+r+s)
pairs = [("Jack","Mary"),("Jack","Jim"),("Jim","Mary")]
def table(B):
    rows=[]
    for i,j in pairs:
        q,r,s,t,J = jaccard_detail(B.loc[i],B.loc[j]); rows.append((f"({i}, {j})",q,r,s,t,f"{q}/({q}+{r}+{s})",round(J,3)))
    return pd.DataFrame(rows,columns=["Pair","q","r","s","t (ignored)","J = q/(q+r+s)","Jaccard"])
main = table(B); main

,Pair,q,r,s,t (ignored),J = q/(q+r+s),Jaccard
0,"(Jack, Mary)",2,0,1,3,2/(2+0+1),0.667
1,"(Jack, Jim)",1,1,1,3,1/(1+1+1),0.333
2,"(Jim, Mary)",1,1,2,2,1/(1+1+2),0.250


## 3. Cross-check against scikit-learn and compare with the simple matching coefficient

In [4]:
from sklearn.metrics import jaccard_score
chk=[(f"({i}, {j})", round(jaccard_score(B.loc[i],B.loc[j]),3), round((B.loc[i]==B.loc[j]).mean(),3)) for i,j in pairs]
pd.DataFrame(chk,columns=["Pair","sklearn jaccard_score","Simple matching (counts 0-0 matches)"])

,Pair,sklearn jaccard_score,Simple matching (counts 0-0 matches)
0,"(Jack, Mary)",0.667,0.833
1,"(Jack, Jim)",0.333,0.667
2,"(Jim, Mary)",0.250,0.500


## 4. Sensitivity: what if the code A means 1?

In [5]:
alt = table(encode(df, positive=("Y","P","A"))); alt

,Pair,q,r,s,t (ignored),J = q/(q+r+s),Jaccard
0,"(Jack, Mary)",2,1,2,1,2/(2+1+2),0.400
1,"(Jack, Jim)",2,1,1,2,2/(2+1+1),0.500
2,"(Jim, Mary)",1,2,3,0,1/(1+2+3),0.167


In [6]:
for name,t in [("A = 0 (main)",main),("A = 1 (alternative)",alt)]:
    best=t.loc[t.Jaccard.idxmax()]
    print(f"{name:22s} most similar pair: {best.Pair}  J = {best.Jaccard}")

A = 0 (main)           most similar pair: (Jack, Mary)  J = 0.667
A = 1 (alternative)    most similar pair: (Jack, Jim)  J = 0.5


## Results and interpretation
| Pair | q | r | s | Jaccard (A = 0) |
|---|---|---|---|---|
| (Jack, Mary) | 2 | 0 | 1 | **2/3 = 0.667** |
| (Jack, Jim) | 1 | 1 | 1 | **1/3 = 0.333** |
| (Jim, Mary) | 1 | 1 | 2 | **1/4 = 0.250** |

* Jack and Mary are the most similar patients: they share a fever and a positive Test-1, and Mary differs only by an extra positive Test-3. Jim and Mary are the least similar.
* Jaccard ignores shared negatives (t). That is appropriate here because two people both testing negative for a rare condition says little about their similarity; the simple matching coefficient, which counts those, inflates every similarity.
* The conclusion depends on how *A* is coded. If A means 1, the ranking changes and (Jack, Jim) becomes the most similar pair (J = 0.5) and (Jim, Mary) the least (J = 0.167). In practice the data dictionary should be checked before clustering.
* 1 − J gives the Jaccard distance used by hierarchical clustering and by k-medoids-style methods on binary data.